Day4 
Tokeniing with code

In [3]:
import tiktoken
encoding = tiktoken.encoding_for_model('gpt-4.1-mini')
tokens = encoding.encode("I love coding in python")

In [4]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"Token ID: {token_id}, Token Text: '{token_text}'")

Token ID: 40, Token Text: 'I'
Token ID: 3047, Token Text: ' love'
Token ID: 22458, Token Text: ' coding'
Token ID: 306, Token Text: ' in'
Token ID: 22752, Token Text: ' python'



And another topic!
The Illusion of "memory"
Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # Load environment variables from .env file
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!



You should be very comfortable with what the next cell is doing!
I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers

In [6]:
from openai import OpenAI
openai = OpenAI()

A message to OpenAI is a list of dicts

In [7]:
messages = [
     {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I love python!"} 
]

In [8]:
responses = openai.chat.completions.create(
    model="gpt-4.1-mini",messages=messages
)
responses.choices[0].message.content

"Hi there! That's awesome—Python is a fantastic language. What do you like most about Python? Are you working on any projects right now?"


OK let's now ask a follow-up question

In [9]:

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's I love?"}
    ]

In [10]:

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'"I love" is a phrase commonly used to express a strong feeling of affection, fondness, or passion towards someone or something. It can refer to romantic feelings, deep appreciation, or enthusiasm. For example:\n\n- "I love my family."\n- "I love reading books."\n- "I love you."\n\nIf you meant something else by "What\'s I love?" please provide more context!'

Wait, wha??
We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "I love python!"},
    {"role": "assistant", "content": "Hi there! That's awesome—Python is a fantastic language. What do you like most about Python? Are you working on any projects right now?"},
    {"role": "user", "content": "What I love?"}
    ]

In [14]:

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

"That's a great question! People often love Python because of its:\n\n1. **Simplicity and Readability**: Python’s clean and straightforward syntax makes it easy to learn and write.\n2. **Versatility**: It’s used in many fields like web development, data science, automation, artificial intelligence, and more.\n3. **Large Community and Libraries**: There’s a huge community and many libraries that make coding faster and easier.\n4. **Cross-Platform**: Python works on Windows, macOS, and Linux.\n5. **Great for Beginners and Experts**: Whether you’re new to programming or an experienced developer, Python is very welcoming.\n\nWhat about you? What do you love most about Python?"

To recap
With apologies if this is obvious to you - but it's still good to reinforce:

Every call to an LLM is stateless
We pass in the entire conversation so far in the input prompt, every time
This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
But this is a trick; it's a by-product of providing the entire conversation, every time
An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!
The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!